# Foundation 01 — Agent Security Architecture and Trust Boundaries

Build an observable boundary inventory for a research-and-support agent, then prove that cross-tenant evidence, model-asserted identity, unregistered paths, dependency failures, missing effect grants, and replay fail closed. The lab is credential-free and the model never becomes the security authority.

![Research-and-support agent trust boundaries](architecture.svg)

Nine numbered boundaries separate untrusted requests, application policy, probabilistic reasoning, data and enterprise systems, decision evidence, and operator recovery.

## 1. Load the course lab

The notebook imports the reusable course module rather than copying its security logic.

In [ ]:
import runpy
from datetime import datetime, timedelta, timezone
from dataclasses import replace
import sys
sys.path.insert(0, '.')
ns = runpy.run_path('lab.py')
ActorContext, BoundaryController, DependencyState, GrantRegistry, Operation = (ns[name] for name in ('ActorContext','BoundaryController','DependencyState','GrantRegistry','Operation'))
build_catalog, request_for, issue_demo_grant, evaluate_controls, unsafe_schema_only_baseline = (ns[name] for name in ('build_catalog','request_for','issue_demo_grant','evaluate_controls','unsafe_schema_only_baseline'))
REQUIRED_BOUNDARIES = ns['REQUIRED_BOUNDARIES']
now = datetime(2026,9,20,12,0,tzinfo=timezone.utc)
actor = ActorContext('user:7','north',frozenset({'request:submit','evidence:read','agent:invoke','ticket:propose','tool:invoke','enterprise:access','memory:read','telemetry:write','agent:operate'}))
catalog = build_catalog()
controller = BoundaryController(catalog)

## 2. Establish the safe baseline

Observe the trusted inputs and the decision evidence before injecting failures.

In [ ]:
valid = request_for('B2-gateway-context',Operation.BUILD_CONTEXT,request_id='nb-valid')
allowed = controller.cross(actor,valid,now=now)
assert unsafe_schema_only_baseline(valid)
assert allowed.allowed and allowed.reason == 'policy-allow'
{'schema_only_baseline': True, 'controlled_decision': allowed.reason, 'boundary': allowed.boundary_id, 'evidence_count': allowed.evidence_count}

## 3. Inject an attack

Change one security-relevant boundary and keep the rest of the fixture stable.

In [ ]:
cross_tenant_request = replace(valid,request_id='nb-cross-tenant',resource_tenant='south')
baseline_allows_attack = unsafe_schema_only_baseline(cross_tenant_request)
cross_tenant = controller.cross(actor,cross_tenant_request,now=now)
claimed_admin = request_for('B4-model-runtime',Operation.PROPOSE_TICKET,request_id='nb-claimed-admin',claimed_subject='admin')
limited = replace(actor,scopes=frozenset({'agent:invoke'}))
identity_attack = controller.cross(limited,claimed_admin,now=now)
assert baseline_allows_attack
assert not cross_tenant.allowed and cross_tenant.reason == 'tenant'
assert not identity_attack.allowed and identity_attack.reason == 'scope'
{'unsafe_baseline_allowed_cross_tenant': baseline_allows_attack, 'controlled_decisions': (cross_tenant, identity_attack)}

## 4. Attempt a bypass

The assertions below make the security property executable and regression-testable.

In [ ]:
direct = controller.cross(actor,replace(valid,request_id='nb-direct',boundary_id='B0-direct-enterprise'),now=now)
policy_down = controller.cross(actor,replace(valid,request_id='nb-policy-down'),state=DependencyState(policy_available=False),now=now)
assert direct.reason == 'unregistered-boundary'
assert policy_down.reason == 'policy-unavailable'
assert all(item.policy_version for item in (direct,policy_down))
(direct, policy_down)

## 5. Evaluate observable outcomes

Use explicit denominators or counts. Private model reasoning is neither required nor recorded.

In [ ]:
report, decisions = evaluate_controls()
assert report.inventory_coverage == report.observed_boundary_coverage == 1
assert report.unexpected_allow_rate == 0
assert report.valid_task_success_rate == report.trace_completeness_rate == 1
{'populations': {'cases': report.cases, 'negative_cases': report.negative_cases, 'attack_cases': report.attack_cases, 'dependency_failure_cases': report.dependency_failure_cases, 'valid_cases': report.valid_cases, 'required_boundaries': report.expected_boundaries}, 'rates': {'inventory': report.inventory_coverage, 'observed': report.observed_boundary_coverage, 'unexpected_allow': report.unexpected_allow_rate, 'valid_task_success': report.valid_task_success_rate, 'trace_completeness': report.trace_completeness_rate}}

## 6. Exercise a second failure mode

In [ ]:
effect = request_for('B5-runtime-tool',Operation.CREATE_TICKET,request_id='nb-effect',evidence_ids=(),logical_operation_id='ticket:case:42:notebook')
missing = controller.cross(actor,effect,now=now)
disabled = controller.cross(actor,replace(effect,request_id='nb-disabled'),state=DependencyState(effects_enabled=False),now=now)
assert missing.reason == 'effect-grant-required' and disabled.reason == 'effects-disabled'
(missing, disabled)

## 7. Exact, expiring, single-use effect grant

The local HMAC receipt models exact binding and atomic consumption. It is a teaching analogue, not production key management or a distributed transaction.

In [ ]:
grant = issue_demo_grant(effect,actor,grant_id='grant:nb',expires_at=now+timedelta(minutes=5))
granted_controller = BoundaryController(catalog,GrantRegistry({grant.grant_id: grant}))
altered_operation = granted_controller.cross(actor,replace(effect,request_id='nb-altered-operation',logical_operation_id='ticket:case:42:changed',effect_grant_id=grant.grant_id),now=now)
granted = granted_controller.cross(actor,replace(effect,request_id='nb-granted',effect_grant_id=grant.grant_id),now=now)
replay = granted_controller.cross(actor,replace(effect,request_id='nb-replay',effect_grant_id=grant.grant_id),now=now)
assert altered_operation.reason == 'invalid-or-replayed-effect-grant'
assert granted.allowed and replay.reason == 'invalid-or-replayed-effect-grant'
(altered_operation, granted, replay)

## 8. Inventory and observation are different claims

A complete catalog proves that required records exist. Observation coverage proves each required boundary produced a decision during this fixture. Neither alone proves security.

In [ ]:
inventory = catalog.audit(REQUIRED_BOUNDARIES)
observed = BoundaryController(catalog).observed_coverage(REQUIRED_BOUNDARIES)
assert inventory['coverage'] == 1 and observed['coverage'] == 0
{'inventory': inventory, 'before_exercise': observed}

## 9. OpenTelemetry SDK: bounded decision attributes

The companion uses the pinned OpenTelemetry Python SDK and an in-memory exporter. Policy executes first; telemetry receives only an allowlisted projection.

In [ ]:
otel = runpy.run_path('otel_adapter.py')
otel_report, spans = otel['demo']()
attrs = [dict(span.attributes) for span in spans]
assert spans and all(set(item) == otel['ALLOWED_ATTRIBUTES'] for item in attrs)
assert 'user:7' not in repr(attrs) and 'case:42' not in repr(attrs)
{'span_count': len(spans), 'first_span': attrs[0]}

## 10. Production replacement

Production replacement: enterprise human and workload identity; signed/versioned policy distribution; complete service, account, region, queue, cache, secret, model-provider, MCP, memory, and effect inventories; resource-level authorization before retrieval; exact approval/grant services; durable idempotency and effect reconciliation; verified revocation propagation; protected OTLP pipelines; append-only evidence; incident owners; and continuous tests that compare deployed routes with the declared inventory. The Python HMAC, list, lock, and in-memory spans prove local invariants only.

## 11. Exercises

1. Add a model-provider boundary without treating provider authentication as user authorization.
2. Add a nested evidence resource and independently authorize its owner.
3. Change one declared route and prove the catalog rejects the mismatch.
4. Design a safe read-only degradation mode and state its maximum policy staleness.
5. Add a production inventory field for owner, recovery objective, and data residency.
6. Compare OpenAI Agents SDK, LangGraph, Google ADK, or Microsoft Agent Framework hooks with the same application-owned boundary contract.

## Checkpoint

Explain which trusted component enforces the invariant, what evidence proves the decision, and what residual risk remains.